# Notebook Bases de Datos Avanzadas

En este notebook se explorarán los datos del dataset de las postulaciones a la **Universidad**, datos obtenidos desde DEMRE.

Veremos:
* Limpieza de datos
* Distribución de datos para generar particiones adecuadas
* Creacion de tablas CQL
* Consultas CQL

## 1. Dataset y análisis

In [11]:
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import uuid

In [13]:
dataset = pd.read_excel("postulaciones.xlsx")
dataset.columns = dataset.columns.str.strip()
dataset.head()

,CEDULA,PERIODO,SEXO,PREFERENCIA,CARRERA,MATRICULADO,FACULTAD,PUNTAJE,GRUPO_DEPEN,REGION,LATITUD,LONGITUD,PTJE_NEM,PSU_PROMLM,PACE,GRATUIDAD
0,15998050,2020,MASCULINO,3,PEDAGOGÍA EN RELIGIÓN Y FILOSOFÍA,NO,CIENCIAS RELIGIOSAS Y FILOSOFICAS,50740,MUNICIPAL,MAULE,-35.090,-71.279,431,5155,NaN,NO
1,17746251,2020,MASCULINO,4,PEDAGOGÍA EN RELIGIÓN Y FILOSOFÍA,NO,CIENCIAS RELIGIOSAS Y FILOSOFICAS,50320,MUNICIPAL,LIBERTADOR GENERAL BERNARDO O'HIGGINS,-34.584,-70.987,445,5320,NaN,NO
2,17825196,2020,MASCULINO,2,KINESIOLOGÍA,NO,CIENCIAS DE LA SALUD,61110,MUNICIPAL,MAULE,-35.339,-72.414,620,5510,NaN,NO
3,18682795,2020,MASCULINO,3,KINESIOLOGÍA,SI,CIENCIAS DE LA SALUD,65585,PARTICULAR SUBVENCIONADO,MAULE,-34.976,-71.224,672,5940,PACE,SI
4,18988817,2020,FEMENINO,1,KINESIOLOGÍA,NO,CIENCIAS DE LA SALUD,60950,MUNICIPAL,LIBERTADOR GENERAL BERNARDO O'HIGGINS,-34.584,-70.987,672,4775,NaN,NO


Analizamos si existen **datos nulos**, es decir, filas nulas por columna

In [14]:
print(f"Cantidad de datos TOTAL: {len(dataset)}")

dataset.isnull().sum()

Cantidad de datos TOTAL: 16651


CEDULA             0
PERIODO            0
SEXO               0
PREFERENCIA        0
CARRERA            0
MATRICULADO        0
FACULTAD           0
PUNTAJE            0
GRUPO_DEPEN        0
REGION             0
LATITUD            0
LONGITUD           0
PTJE_NEM           0
PSU_PROMLM         0
PACE           16582
GRATUIDAD          0
dtype: int64

Básicamente casi todos tienen el PACE nulo, podríamos eliminar esa columna.

Verificamos **datos duplicados**

In [15]:
# dataset.duplicated().sum()

dataset['CEDULA'].duplicated().sum()

np.int64(5656)

No hay datos duplicados.

## 2. Conexión con Cassandra y creación de KeySpace



El Keyspace tiene **factor de replicación de 3** y utiliza **SimpleStrategy**

In [16]:
%pip install cassandra-driver

Note: you may need to restart the kernel to use updated packages.


Levantar el cluster antes de ejecutar la siguiente celda

In [17]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement
from cassandra import ConsistencyLevel

cluster = Cluster(["127.0.0.1"], port = 19042)
session = cluster.connect()

In [18]:
session.execute("""

CREATE KEYSPACE IF NOT EXISTS inf325grupo2 
WITH replication = {
    'class': 'SimpleStrategy',
    'replication_factor': 3
};

""")

Usamos el **Keyspace**

In [19]:
session.set_keyspace("inf325grupo2")

## 3. Creación de tablas CQL


Cada tabla se creará en base a una consulta de negocio de las 3 definidas en el enunciado del laboratorio.

Para cada caso analizaremos:
* La consulta a realizar
* La Row-Key a utilizar
* Cuántas particiones genera esa RK
* Qué tan balanceada es la distribución de datos por partición

> Borramos las tablas si es que existen, para evitar problemas con datos anteriores

In [20]:
session.execute("DROP TABLE IF EXISTS medicina_por_periodo;")
session.execute("DROP TABLE IF EXISTS informatica_maule_por_periodo;")
session.execute("DROP TABLE IF EXISTS salud_por_puntaje;")

### 3.1 Consulta 1: 

Devolver todos los postulantes **matriculados** en la **carrera** de medicina ordenados por **periodo**.

> Row Key = ((matriculado, carrera), periodo, id)

Analizamos la **distribución de datos** para generar particiones adecuadas

In [21]:
particiones = dataset.groupby(["MATRICULADO", "CARRERA"]).size()

print(particiones)

MATRICULADO  CARRERA                                                  
NO           AGRONOMÍA                                                     251
             AUDITORIA (CURICO)                                             51
             AUDITORÍA                                                     155
             CONSTRUCCIÓN CIVIL                                             47
             EDUCACIÓN ESPECIAL                                            282
             EDUCACIÓN PARVULARIA                                           94
             ENFERMERÍA                                                   2350
             ENFERMERÍA (CURICO)                                          1377
             INGENIERIA CIVIL                                              166
             INGENIERIA CIVIL INDUSTRIAL                                   296
             INGENIERÍA CIVIL INFORMÁTICA                                  301
             INGENIERÍA COMERCIAL                           

In [22]:
particiones.describe()

count      59.000000
mean      282.220339
std       470.025204
min        10.000000
25%        88.000000
50%       132.000000
75%       250.000000
max      2422.000000
dtype: float64

* **Número de particiones:** 59
* **Desviación estándar de datos por partición:** 470
* **Rango de cantidad de datos por partición:** 10 a 2422

In [23]:
session.execute("""
CREATE TABLE IF NOT EXISTS medicina_por_periodo (
    region TEXT,
    periodo INT,
    cedula BIGINT,
    sexo TEXT,
    preferencia INT,
    carrera TEXT,
    matriculado TEXT,
    facultad TEXT,
    puntaje INT,
    grupo_depen TEXT,
    latitud DOUBLE,
    longitud DOUBLE,
    ptje_nem INT,
    psu_promlm INT,
    pace TEXT,
    gratuidad TEXT,
    id UUID,
    PRIMARY KEY ((matriculado, carrera), periodo, id)
)WITH CLUSTERING ORDER BY (periodo ASC);
"""
)

### 3.2 Consulta 2:

Devolver todos los postulantes **matriculados** provenientes de la **región** del Maule en la **carrera** Ingeniería Civil Informática ordenados por **periodo**.

> Row Key = ((region, matriculado, carrera), periodo, id)

In [24]:
particiones = dataset.groupby(["MATRICULADO", "REGION", "CARRERA"]).size()

print(particiones)

MATRICULADO  REGION       CARRERA                      
NO           ANTOFAGASTA  AGRONOMÍA                        1
                          ENFERMERÍA                       6
                          ENFERMERÍA (CURICO)              4
                          INGENIERIA CIVIL                 1
                          INGENIERIA CIVIL INDUSTRIAL      1
                                                          ..
SI           VALPARAÍSO   AGRONOMÍA                        1
                          ENFERMERÍA (CURICO)              1
                          MEDICINA                         9
                          PEDAGOGÍA EN EDUCACIÓN FÍSICA    1
                          PEDAGOGÍA EN INGLÉS              1
Length: 327, dtype: int64


In [25]:
particiones.describe()

count     327.000000
mean       50.920489
std       143.951700
min         1.000000
25%         1.000000
50%         7.000000
75%        33.500000
max      1624.000000
dtype: float64

* **Número de particiones:** 327
* **Desviación estándar de datos por partición:** 144
* **Rango de cantidad de datos por partición:** 1 a 1624

In [26]:
session.execute("""
CREATE TABLE IF NOT EXISTS informatica_maule_por_periodo (
    region TEXT,
    periodo INT,
    cedula BIGINT,
    sexo TEXT,
    preferencia INT,
    carrera TEXT,
    matriculado TEXT,
    facultad TEXT,
    puntaje INT,
    grupo_depen TEXT,
    latitud DOUBLE,
    longitud DOUBLE,
    ptje_nem INT,
    psu_promlm INT,
    pace TEXT,
    gratuidad TEXT,
    id UUID,
    PRIMARY KEY ((matriculado, region, carrera), periodo, id)
) WITH CLUSTERING ORDER BY (periodo ASC);
"""
)

### 3.3 Consulta 3:

Devolver todos los postulantes **matriculados** en la **facultad** de Ciencias de la Salud ordenado por **puntaje PSU**.

> Row Key = ((matriculado, facultad), puntaje, id)

In [27]:
particiones = dataset.groupby(["MATRICULADO", "FACULTAD"]).size()

print(particiones)

MATRICULADO  FACULTAD                         
NO           CIENCIAS AGRARIAS Y FORESTALES        522
             CIENCIAS BASICAS                      212
             CIENCIAS DE LA EDUCACION             1237
             CIENCIAS DE LA INGENIERIA            1168
             CIENCIAS DE LA SALUD                 6472
             CIENCIAS RELIGIOSAS Y FILOSOFICAS      28
             CIENCIAS SOCIALES Y ECONOMICAS       1304
             MEDICINA                             2422
SI           CIENCIAS AGRARIAS Y FORESTALES        246
             CIENCIAS BASICAS                      123
             CIENCIAS DE LA EDUCACION              898
             CIENCIAS DE LA INGENIERIA             576
             CIENCIAS DE LA SALUD                  825
             CIENCIAS RELIGIOSAS Y FILOSOFICAS      15
             CIENCIAS SOCIALES Y ECONOMICAS        421
             MEDICINA                              182
dtype: int64


In [28]:
particiones.describe()

count      16.000000
mean     1040.687500
std      1579.924923
min        15.000000
25%       204.500000
50%       549.000000
75%      1185.250000
max      6472.000000
dtype: float64

* **Número de particiones:** 16
* **Desviación estándar de datos por partición:** 1580
* **Rango de cantidad de datos por partición:** 15 a 6427

In [29]:
session.execute("""
CREATE TABLE IF NOT EXISTS salud_por_puntaje (
    region TEXT,
    periodo INT,
    cedula BIGINT,
    sexo TEXT,
    preferencia INT,
    carrera TEXT,
    matriculado TEXT,
    facultad TEXT,
    puntaje INT,
    grupo_depen TEXT,
    latitud DOUBLE,
    longitud DOUBLE,
    ptje_nem INT,
    psu_promlm INT,
    pace TEXT,
    gratuidad TEXT,
    id UUID,
    PRIMARY KEY ((matriculado, facultad), puntaje, id)
) WITH CLUSTERING ORDER BY (puntaje ASC);
"""
)

## 4. Rellenar las tablas con los datos del dataset


### 4.1 Población de tabla consulta 1


In [30]:
# Creamos la query para poblar la tabla
query_consulta1 = session.prepare(
    """
    INSERT INTO medicina_por_periodo(
        region, periodo, cedula, sexo, preferencia, carrera, matriculado, facultad, 
        puntaje, grupo_depen, latitud, longitud, ptje_nem, psu_promlm, pace, gratuidad, id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
)

# Recorremos el dataset y cargamos los datos
for index, row in dataset.iterrows():
    session.execute(query_consulta1, (
        str(row['REGION']),
        int(row['PERIODO']),
        int(row['CEDULA']),
        str(row['SEXO']),
        int(row['PREFERENCIA']),
        str(row['CARRERA']),
        str(row['MATRICULADO']),
        str(row['FACULTAD']),
        int(row['PUNTAJE']),
        str(row['GRUPO_DEPEN']),
        float(row['LATITUD']),
        float(row['LONGITUD']),
        int(row['PTJE_NEM']),
        int(row['PSU_PROMLM']),
        str(row['PACE']) if pd.notnull(row['PACE']) else "N/A",
        str(row['GRATUIDAD']),
        uuid.uuid4()
    ))


### 4.2 Población de tabla consulta 2

In [40]:
# Creamos la query para poblar la tabla
query_consulta1 = session.prepare(
    """
    INSERT INTO informatica_maule_por_periodo(
        region, periodo, cedula, sexo, preferencia, carrera, matriculado, facultad, 
        puntaje, grupo_depen, latitud, longitud, ptje_nem, psu_promlm, pace, gratuidad, id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
)

# Recorremos el dataset y cargamos los datos
for index, row in dataset.iterrows():
    session.execute(query_consulta1, (
        str(row['REGION']),
        int(row['PERIODO']),
        int(row['CEDULA']),
        str(row['SEXO']),
        int(row['PREFERENCIA']),
        str(row['CARRERA']),
        str(row['MATRICULADO']),
        str(row['FACULTAD']),
        int(row['PUNTAJE']),
        str(row['GRUPO_DEPEN']),
        float(row['LATITUD']),
        float(row['LONGITUD']),
        int(row['PTJE_NEM']),
        int(row['PSU_PROMLM']),
        str(row['PACE']) if pd.notnull(row['PACE']) else "N/A",
        str(row['GRATUIDAD']),
        uuid.uuid4()
    ))

### 4.3 Población de tabla consulta 3.

In [ ]:
# Creamos la query para poblar la tabl
query_consulta1 = session.prepare(
    """
    INSERT INTO salud_por_puntaje(
        region, periodo, cedula, sexo, preferencia, carrera, matriculado, facultad, 
        puntaje, grupo_depen, latitud, longitud, ptje_nem, psu_promlm, pace, gratuidad, id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
)

# Recorremos el dataset y cargamos los datos
for index, row in dataset.iterrows():
    session.execute(query_consulta1, (
        str(row['REGION']),
        int(row['PERIODO']),
        int(row['CEDULA']),
        str(row['SEXO']),
        int(row['PREFERENCIA']),
        str(row['CARRERA']),
        str(row['MATRICULADO']),
        str(row['FACULTAD']),
        int(row['PUNTAJE']),
        str(row['GRUPO_DEPEN']),
        float(row['LATITUD']),
        float(row['LONGITUD']),
        int(row['PTJE_NEM']),
        int(row['PSU_PROMLM']),
        str(row['PACE']) if pd.notnull(row['PACE']) else "N/A",
        str(row['GRATUIDAD']),
        uuid.uuid4()
    ))

## 5. Consultas a la Base de Datos

### 5.1 Consulta 1

In [ ]:
rows = session.execute("""
SELECT *
FROM medicina_por_periodo
WHERE matriculado = 'SI'
AND carrera = 'MEDICINA';
""")
resultado = pd.DataFrame(rows)
resultado

InvalidRequest: Error from server: code=2200 [Invalid query] message="No keyspace has been specified. USE a keyspace, or explicitly specify keyspace.tablename"

### 5.2 Consulta 2

In [ ]:

rows = session.execute("""
SELECT * FROM informatica_maule_por_periodo WHERE carrera = 'INGENIERÍA CIVIL INFORMÁTICA' AND region = 'MAULE' AND matriculado = 'SI';
""")
resultado = pd.DataFrame(rows)
resultado

,matriculado,region,carrera,periodo,id,cedula,facultad,gratuidad,grupo_depen,latitud,longitud,pace,preferencia,psu_promlm,ptje_nem,puntaje,sexo
0,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2018,124319d8-fc8a-40af-b467-0e9de8d302c9,19215629,CIENCIAS DE LA INGENIERIA,NO,MUNICIPAL,-35.083,-72.017,N/A,1,6045,663,64465,MASCULINO
1,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2018,1bc22cc4-7e13-4517-9f5e-1a227c9b107b,19389013,CIENCIAS DE LA INGENIERIA,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,2,5415,556,56415,MASCULINO
2,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2018,1cba2cc0-571a-4dcf-b513-f6162e1bb9ad,19541641,CIENCIAS DE LA INGENIERIA,NO,MUNICIPAL,-35.083,-72.017,N/A,1,4955,635,56975,MASCULINO
3,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2018,1cbce070-6a5f-4b84-b22f-782794fda736,19039481,CIENCIAS DE LA INGENIERIA,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,1,6750,684,68710,MASCULINO
4,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2018,272a044b-90fb-4668-8587-dee638bae8f6,18681691,CIENCIAS DE LA INGENIERIA,NO,MUNICIPAL,-34.976,-71.224,N/A,2,5450,449,50640,MASCULINO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2020,e8308b19-be62-488f-96e2-5bf240b3d8de,19923015,CIENCIAS DE LA INGENIERIA,SI,PARTICULAR SUBVENCIONADO,-34.976,-71.224,N/A,1,5925,528,55800,MASCULINO
88,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2020,e8ae7c9d-168d-4b72-9f27-1c58ab8c5399,19808137,CIENCIAS DE LA INGENIERIA,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,1,6110,647,62130,MASCULINO
89,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2020,f6f66255-4ef4-4679-ab93-58a1503d4efd,19808090,CIENCIAS DE LA INGENIERIA,SI,MUNICIPAL,-35.423,-71.657,PACE,1,6550,667,66980,MASCULINO
90,SI,MAULE,INGENIERÍA CIVIL INFORMÁTICA,2020,fa1b3937-d146-4718-9067-7523c586a205,19959090,CIENCIAS DE LA INGENIERIA,SI,MUNICIPAL,-35.217,-71.250,N/A,1,5570,617,61230,FEMENINO


### 5.3 Consulta 3

In [ ]:
rows = session.execute("""
SELECT * FROM salud_por_puntaje WHERE facultad = 'CIENCIAS DE LA SALUD' AND matriculado = 'SI';
""")
resultado = pd.DataFrame(rows)
resultado

,matriculado,facultad,puntaje,id,carrera,cedula,gratuidad,grupo_depen,latitud,longitud,pace,periodo,preferencia,psu_promlm,ptje_nem,region,sexo
0,SI,CIENCIAS DE LA SALUD,52100,167cb25b-262a-4357-b036-0f67eb2b8c21,NUTRICIÓN Y DIETÉTICA,19300014,NO,PARTICULAR SUBVENCIONADO,-34.976,-71.224,N/A,2019,2,5435,507,MAULE,FEMENINO
1,SI,CIENCIAS DE LA SALUD,52240,fd1aeca4-2935-49ce-9b51-de4f1814b1cf,KINESIOLOGÍA,19045497,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,2018,2,4940,550,MAULE,MASCULINO
2,SI,CIENCIAS DE LA SALUD,52530,106407af-4d69-4fe2-aad7-5342b296a04f,PSICOLOGÍA,19805433,NO,MUNICIPAL,-35.423,-71.657,N/A,2018,2,5260,548,MAULE,FEMENINO
3,SI,CIENCIAS DE LA SALUD,52580,39d16196-d5b9-45ae-bf6c-a221cea4ae54,KINESIOLOGÍA,19650446,SI,PARTICULAR SUBVENCIONADO,-35.339,-72.414,N/A,2019,1,5530,492,MAULE,FEMENINO
4,SI,CIENCIAS DE LA SALUD,53000,a02fdde7-6d4d-487c-8623-3069d227866b,NUTRICIÓN Y DIETÉTICA,19390264,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,2018,6,5345,532,MAULE,FEMENINO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
820,SI,CIENCIAS DE LA SALUD,72970,f1617e21-b8f4-4fb0-a8ac-9f056d5d3c5a,ENFERMERÍA,19697311,NO,PARTICULAR PAGADO,-37.621,-73.456,N/A,2020,3,6910,740,BÍO-BÍO,FEMENINO
821,SI,CIENCIAS DE LA SALUD,73670,3f2da0af-aed9-4f6f-a2c5-b307d673501e,ENFERMERÍA,19928091,SI,PARTICULAR SUBVENCIONADO,-34.983,-72.000,N/A,2020,2,6490,770,MAULE,FEMENINO
822,SI,CIENCIAS DE LA SALUD,73730,2507f1f6-32da-4101-ac6f-35983c0af362,ENFERMERÍA,19697810,NO,MUNICIPAL,-35.516,-71.572,N/A,2020,4,6810,754,MAULE,FEMENINO
823,SI,CIENCIAS DE LA SALUD,73870,3fe0c342-bbcf-4d0d-9249-7c6fa05abda0,ENFERMERÍA,19696770,SI,MUNICIPAL,-35.423,-71.657,N/A,2020,3,6740,754,MAULE,FEMENINO


In [ ]:
condicion_valores = (dataset['MATRICULADO'] == 'SI') & (dataset['FACULTAD'] == 'CIENCIAS DE LA SALUD')

# 2. Definimos la condición de duplicados
condicion_duplicados = dataset.duplicated(subset=['MATRICULADO', 'FACULTAD', 'CEDULA', 'PUNTAJE'], keep=False)

# 3. Combinamos ambas para obtener el resultado final
duplicados_filtrados = dataset[condicion_valores & condicion_duplicados]

# Visualizar el resultado
duplicados_filtrados

,CEDULA,PERIODO,SEXO,PREFERENCIA,CARRERA,MATRICULADO,FACULTAD,PUNTAJE,GRUPO_DEPEN,REGION,LATITUD,LONGITUD,PTJE_NEM,PSU_PROMLM,PACE,GRATUIDAD
12303,19389165,2018,FEMENINO,2,ENFERMERÍA,SI,CIENCIAS DE LA SALUD,62540,PARTICULAR PAGADO,MAULE,-35.423,-71.657,692,5620,NaN,NO
15151,19389165,2018,FEMENINO,3,ENFERMERÍA (CURICO),SI,CIENCIAS DE LA SALUD,62540,PARTICULAR PAGADO,MAULE,-35.423,-71.657,692,5620,NaN,NO


## 6. Pruebas de Consistencia y Disponibilidad

Una de las caracteristicas de Cassandra es que es una base de datos distribuida, por ende, debe contar con cierto nivel de consistencia y disponibilidad, para ello realizaremos un par de pruebas en donde se demostrara el desempeño del cluster en estas tareas.

### 6.1 Prueba de Consistencia (Metodo THREE)

Como primera prueba de consistencia insertaremos, consultaremos y eliminaremos en el cluster de cassandra una fila de prueba utilizando el metodo THREE en la query.

> Nos conectamos a la BD e insertamos una fila de prueba con el metodo three

In [ ]:
cluster = Cluster(["127.0.0.1"], port = 19042)
session = cluster.connect()
session.set_keyspace("inf325grupo2")

query_consistenciaTHREE = SimpleStatement(
    """
    INSERT INTO informatica_maule_por_periodo(
        region, periodo, cedula, sexo, preferencia, carrera, matriculado, facultad, 
        puntaje, grupo_depen, latitud, longitud, ptje_nem, psu_promlm, pace, gratuidad, id
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,
    consistency_level=ConsistencyLevel.THREE
)

session.execute(query_consistenciaTHREE, (
    "test",
    2026,
    1,
    "test",
    1,
    "test",
    "test",
    "test",
    1,
    "test",
    1.0,
    1.0,
    1,
    1,
    "test",
    "test",
    uuid.uuid4()
))

> Consultamos por la fila de prueba recien insertada con el metodo three

In [ ]:
query_consistenciaTHREE = SimpleStatement("""
SELECT * FROM informatica_maule_por_periodo WHERE carrera = 'test' AND region = 'test' AND matriculado = 'test';
""", consistency_level=ConsistencyLevel.THREE)
rows = session.execute(query_consistenciaTHREE)
resultado = pd.DataFrame(rows)
resultado

,matriculado,region,carrera,periodo,id,cedula,facultad,gratuidad,grupo_depen,latitud,longitud,pace,preferencia,psu_promlm,ptje_nem,puntaje,sexo
0,test,test,test,2026,21cac8bd-22f7-44a4-b766-7f9046048458,1,test,test,test,1.0,1.0,test,1,1,1,1,test


> Eliminamos la fila de prueba con el metodo three

In [ ]:
query_consistenciaTHREE = SimpleStatement("""
    DELETE FROM informatica_maule_por_periodo WHERE carrera = 'test' AND region = 'test' AND matriculado = 'test';
    """, consistency_level=ConsistencyLevel.THREE)
rows = session.execute(query_consistenciaTHREE)
resultado = pd.DataFrame(rows)
resultado

""


### 6.2 Prueba de Consistencia (Insertar)

Como segunda prueba de consistencia insertaremos en el NODO 1 del cluster de cassandra una fila de prueba para luego recuperarla desde el NODO 3.

> Nos conectamos al NODO 1 e insertamos una fila de prueba.

In [ ]:
cluster = Cluster(["127.0.0.1"], port=9042)
session = cluster.connect()
ip_actual = list(session.hosts)[0]
print("IP NODO ACTUAL: " + str(ip_actual))
session.set_keyspace("inf325grupo2")

query_consistencia1 = session.prepare(
    """
    INSERT INTO informatica_maule_por_periodo(
        region, periodo, cedula, sexo, preferencia, carrera, matriculado, facultad, 
        puntaje, grupo_depen, latitud, longitud, ptje_nem, psu_promlm, pace, gratuidad, id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
)

session.execute(query_consistencia1, (
    "test",
    2026,
    1,
    "test",
    1,
    "test",
    "test",
    "test",
    1,
    "test",
    1.0,
    1.0,
    1,
    1,
    "test",
    "test",
    uuid.uuid4()
))

IP NODO ACTUAL: 127.0.0.1:9042


> Nos conectamos al NODO 3 y consultamos por la fila de prueba

In [ ]:
cluster = Cluster(["127.0.0.1"], port=9044)
session = cluster.connect()
ip_actual = list(session.hosts)[0]
print("IP NODO ACTUAL: " + str(ip_actual))
session.set_keyspace("inf325grupo2")

rows = session.execute("""
SELECT * FROM informatica_maule_por_periodo WHERE carrera = 'test' AND region = 'test' AND matriculado = 'test';
""")
resultado = pd.DataFrame(rows)
resultado

IP NODO ACTUAL: 127.0.0.1:9044


,matriculado,region,carrera,periodo,id,cedula,facultad,gratuidad,grupo_depen,latitud,longitud,pace,preferencia,psu_promlm,ptje_nem,puntaje,sexo
0,test,test,test,2026,48632997-b5d5-4add-a46c-70117bc349d9,1,test,test,test,1.0,1.0,test,1,1,1,1,test


### 6.3 Prueba de Consistencia (Eliminar)

Para esta tercera prueba de consistencia, nos conectaremos al NODO 2 y eliminaremos la fila de prueba anteriormente insertada, luego nos conectaremos al NODO 1 y buscaremos la fila de prueba.

> Nos conectamos al NODO 2 y eliminamos la fila de prueba anterior.

In [ ]:
cluster = Cluster(["127.0.0.1"], port=9043)
session = cluster.connect()
ip_actual = list(session.hosts)[0]
print("IP NODO ACTUAL: " + str(ip_actual))
session.set_keyspace("inf325grupo2")

session.execute(
    """
    DELETE FROM informatica_maule_por_periodo WHERE carrera = 'test' AND region = 'test' AND matriculado = 'test';
    """
)

IP NODO ACTUAL: 127.0.0.1:9043


> Nos conectamos al NODO 1 y buscamos la fila de prueba eliminada.

In [ ]:
cluster = Cluster(["127.0.0.1"], port=9042)
session = cluster.connect()
ip_actual = list(session.hosts)[0]
print("IP NODO ACTUAL: " + str(ip_actual))
session.set_keyspace("inf325grupo2")

rows = session.execute("""
SELECT * FROM informatica_maule_por_periodo WHERE carrera = 'test' AND region = 'test' AND matriculado = 'test';
""")
resultado = pd.DataFrame(rows)
resultado

IP NODO ACTUAL: 127.0.0.1:9042


""


### 6.4 Prueba de Disponibilidad

Para realizar la prueba de disponibilidad es necesario tener instalado y configurado el NGINX como se especifico en el README, la prueba consiste conectarse a cassandra a su primer nodo, realizar una consulta a la base de datos, luego apagar el contenedor del nodo en Docker, y finalizar realizando otra consulta a la base de datos.

> Nos conectamos al primer nodo que retorne Cassandra

In [32]:
cluster = Cluster(["127.0.0.1"], port = 19042)
session = cluster.connect()
session.set_keyspace("inf325grupo2")

> Realizamos la primera consulta

In [33]:
rows = session.execute("""
SELECT *
FROM medicina_por_periodo
WHERE matriculado = 'SI'
AND carrera = 'MEDICINA';
""")
resultado = pd.DataFrame(rows)
resultado.head(5)

,matriculado,carrera,periodo,id,cedula,facultad,gratuidad,grupo_depen,latitud,longitud,pace,preferencia,psu_promlm,ptje_nem,puntaje,region,sexo
0,SI,MEDICINA,2018,01889b4f-ba27-4181-be05-0f9ae4f7ff83,19386431,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,1,7330,768,76135,MAULE,FEMENINO
1,SI,MEDICINA,2018,120d55a0-2105-43f1-8428-745c342f2bde,19132428,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-45.564,-72.065,N/A,2,6770,777,73845,AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO,FEMENINO
2,SI,MEDICINA,2018,12a745a8-b674-4ee1-9655-8226c71a6c8b,19471958,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-35.339,-72.414,N/A,2,7035,768,73225,MAULE,MASCULINO
3,SI,MEDICINA,2018,19d65ef8-0a2f-47f3-b747-b3257d392cdb,19124760,MEDICINA,NO,PARTICULAR PAGADO,-33.066,-71.329,N/A,4,7145,711,73155,VALPARAÍSO,FEMENINO
4,SI,MEDICINA,2018,1bdb4a2f-ddec-43c1-8896-16db3011e644,19388477,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-20.244,-70.139,N/A,1,7115,727,72865,TARAPACÁ,FEMENINO


> Apagamos el contenedor del NODO 1

In [34]:
!docker stop nodo_1

nodo_1


> Realizamos nuevamente la consulta

In [35]:
rows = session.execute("""
SELECT *
FROM medicina_por_periodo
WHERE matriculado = 'SI'
AND carrera = 'MEDICINA';
""")
resultado = pd.DataFrame(rows)
resultado.head(5)

,matriculado,carrera,periodo,id,cedula,facultad,gratuidad,grupo_depen,latitud,longitud,pace,preferencia,psu_promlm,ptje_nem,puntaje,region,sexo
0,SI,MEDICINA,2018,01889b4f-ba27-4181-be05-0f9ae4f7ff83,19386431,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,1,7330,768,76135,MAULE,FEMENINO
1,SI,MEDICINA,2018,120d55a0-2105-43f1-8428-745c342f2bde,19132428,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-45.564,-72.065,N/A,2,6770,777,73845,AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO,FEMENINO
2,SI,MEDICINA,2018,12a745a8-b674-4ee1-9655-8226c71a6c8b,19471958,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-35.339,-72.414,N/A,2,7035,768,73225,MAULE,MASCULINO
3,SI,MEDICINA,2018,19d65ef8-0a2f-47f3-b747-b3257d392cdb,19124760,MEDICINA,NO,PARTICULAR PAGADO,-33.066,-71.329,N/A,4,7145,711,73155,VALPARAÍSO,FEMENINO
4,SI,MEDICINA,2018,1bdb4a2f-ddec-43c1-8896-16db3011e644,19388477,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-20.244,-70.139,N/A,1,7115,727,72865,TARAPACÁ,FEMENINO


In [36]:
!docker stop nodo_2

nodo_2


In [37]:
rows = session.execute("""
SELECT *
FROM medicina_por_periodo
WHERE matriculado = 'SI'
AND carrera = 'MEDICINA';
""")
resultado = pd.DataFrame(rows)
resultado.head(5)

,matriculado,carrera,periodo,id,cedula,facultad,gratuidad,grupo_depen,latitud,longitud,pace,preferencia,psu_promlm,ptje_nem,puntaje,region,sexo
0,SI,MEDICINA,2018,01889b4f-ba27-4181-be05-0f9ae4f7ff83,19386431,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-35.423,-71.657,N/A,1,7330,768,76135,MAULE,FEMENINO
1,SI,MEDICINA,2018,120d55a0-2105-43f1-8428-745c342f2bde,19132428,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-45.564,-72.065,N/A,2,6770,777,73845,AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO,FEMENINO
2,SI,MEDICINA,2018,12a745a8-b674-4ee1-9655-8226c71a6c8b,19471958,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-35.339,-72.414,N/A,2,7035,768,73225,MAULE,MASCULINO
3,SI,MEDICINA,2018,19d65ef8-0a2f-47f3-b747-b3257d392cdb,19124760,MEDICINA,NO,PARTICULAR PAGADO,-33.066,-71.329,N/A,4,7145,711,73155,VALPARAÍSO,FEMENINO
4,SI,MEDICINA,2018,1bdb4a2f-ddec-43c1-8896-16db3011e644,19388477,MEDICINA,NO,PARTICULAR SUBVENCIONADO,-20.244,-70.139,N/A,1,7115,727,72865,TARAPACÁ,FEMENINO


In [38]:
!docker stop nodo_3

nodo_3


In [39]:
rows = session.execute("""
SELECT *
FROM medicina_por_periodo
WHERE matriculado = 'SI'
AND carrera = 'MEDICINA';
""")
resultado = pd.DataFrame(rows)
resultado.head(5)

NoHostAvailable: ('Unable to complete the operation against any hosts', {<Host: 127.0.0.1:19042 dc1>: ConnectionShutdown('Connection to 127.0.0.1:19042 was closed')})